# ActionShap — Milestone-Zero Gate Notebook

**Purpose.** Decide, before any implementation time is spent, whether the revised ActionShap design is buildable as specified. It runs the checks corresponding to the seven corrections in revision 2 of `ActionShap_Recommendation_Spec.md`.

This notebook is a *spike*, not the pipeline. It uses small models, sampled candidate sets, and short training runs. Nothing here produces a number for the paper.

## What it tests

| Part | Question | Spec section |
|---|---|---|
| 3 | Does profile masking actually change the recommender's output? | §7.1.1 |
| 4 | Are the empty and full coalitions well defined and deterministic? | §7.4, §14 |
| 5 | Is the efficiency diagnostic informative, or an identity? | §9 |
| 6 | Is the intervention oracle `a*` computable, and how bad is greedy? | §11.4 |
| 7 | Does AIA have a sane null, and is RQ2 circular at `B=1`? | §11.1 |

## How to run

Runs end to end on CPU in roughly two to five minutes on synthetic data. Needs only `numpy`, `pandas`, `scipy`, and optionally `matplotlib`.

To use real data instead, set `CONFIG["ml1m_ratings_path"]` in the next cell to your `ml-1m/ratings.dat`. Synthetic data is sufficient for every gate here — the questions are structural, not empirical.

## How to read the result

The final cell prints a PASS/FAIL table. **The gate that decides the project is `masking_sensitivity`.** If it fails on the history-conditioned model, the player definition in §6 cannot be paired with the intervention semantics in §8, and the design needs rethinking rather than debugging.

In [ ]:
from __future__ import annotations

import math
import time
from dataclasses import dataclass
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

SEED = 20260802

CONFIG = {
    # Point this at ml-1m/ratings.dat to use real data. Empty string -> synthetic.
    "ml1m_ratings_path": "",
    "n_factors": 32,
    "train_epochs": 40,
    "n_max_history": 20,      # spec 6: n_max, reduced from 50 to keep the spike fast
    "candidate_size": 200,    # spec 7.3
    "ndcg_k": 10,             # spec 7.4
    "n_gate_users": 200,      # spec 7.1.1 requires at least 200
    "n_eval_users": 40,       # users carried into the expensive Shapley/intervention parts
    "mc_permutations": 50,    # doubled by antithetic pairing
    "null_draws": 1000,       # spec 11.1
    "rho_grid": (0.0, 0.25, 0.5),  # spec 8
}

GATES: dict[str, tuple[bool, str]] = {}


def record_gate(name: str, passed: bool, detail: str) -> None:
    """Register a pass/fail outcome and echo it immediately."""
    GATES[name] = (bool(passed), detail)
    print(f"[{'PASS' if passed else 'FAIL'}] {name}: {detail}")


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30.0, 30.0)))


print("numpy", np.__version__, "| pandas", pd.__version__)
print("config:", {k: v for k, v in CONFIG.items() if k != "ml1m_ratings_path"})

## Part 1 — Data, filtering, and the temporal split

Two things here matter beyond just getting data into memory.

**The tie-break rule (spec §7.2).** Interactions are sorted by `(timestamp, original_record_index)` with a stable sort. MovieLens timestamps are second-resolution so ties are rare, but any Amazon-derived secondary dataset has day-resolution timestamps where a user reviewing several items on the same day produces ties constantly. Without a fixed tie-break the identity of the held-out test item changes between runs, which surfaces much later as irreproducible attributions rather than as an obvious data bug.

**The synthetic generator is built to contain redundancy on purpose.** Users draw their interactions from two or three item groups, so within a group the history items are mutually substitutable. That is the structure the `B>=2` comparison in Part 7 needs in order to have anything to detect — if the generator produced independent interactions, leave-one-out and Shapley would agree trivially and the notebook could not tell a working method from a broken one.

In [ ]:
def load_ml1m(path: str) -> pd.DataFrame:
    df = pd.read_csv(
        path, sep="::", engine="python", encoding="latin-1",
        names=["user", "item", "rating", "ts"],
    )
    df = df.loc[df["rating"] >= 4, ["user", "item", "ts"]]
    return df.reset_index(drop=True)


def make_synthetic(n_users=1500, n_items=900, n_groups=15, seed=SEED) -> pd.DataFrame:
    """Group-structured interactions, so history items within a group are redundant."""
    rng = np.random.default_rng(seed)
    item_group = rng.integers(0, n_groups, size=n_items)
    by_group = [np.where(item_group == g)[0] for g in range(n_groups)]
    rows = []
    for u in range(n_users):
        liked = rng.choice(n_groups, size=int(rng.integers(2, 4)), replace=False)
        pool = np.concatenate([by_group[g] for g in liked])
        n_int = int(min(rng.integers(10, 45), pool.size))
        items = rng.choice(pool, size=n_int, replace=False)
        n_noise = int(rng.integers(0, 4))
        if n_noise:
            items = np.concatenate([items, rng.integers(0, n_items, size=n_noise)])
        ts = np.sort(rng.integers(0, 1_000_000, size=items.size))
        rows.extend((u, int(i), int(t)) for i, t in zip(items, ts))
    df = pd.DataFrame(rows, columns=["user", "item", "ts"])
    return df.drop_duplicates(subset=["user", "item"], keep="first").reset_index(drop=True)


def five_core(df: pd.DataFrame, k: int = 5) -> pd.DataFrame:
    """Iterate to a fixed point -- a single pass leaves users below threshold."""
    while True:
        uc, ic = df["user"].value_counts(), df["item"].value_counts()
        out = df[df["user"].isin(uc[uc >= k].index) & df["item"].isin(ic[ic >= k].index)]
        if len(out) == len(df):
            return out.reset_index(drop=True)
        df = out


def reindex(df: pd.DataFrame):
    uids, iids = np.sort(df["user"].unique()), np.sort(df["item"].unique())
    umap = {u: i for i, u in enumerate(uids)}
    imap = {v: i for i, v in enumerate(iids)}
    df = df.assign(user=df["user"].map(umap), item=df["item"].map(imap))
    return df, len(uids), len(iids)


def temporal_split(df: pd.DataFrame):
    """Last interaction -> test, second-last -> validation. Deterministic ties (spec 7.2)."""
    df = df.copy()
    df["row"] = np.arange(len(df))
    df = df.sort_values(["user", "ts", "row"], kind="mergesort").reset_index(drop=True)
    train, val, test = {}, {}, {}
    for u, g in df.groupby("user", sort=True):
        items = g["item"].to_numpy()
        if items.size < 4:
            continue
        train[int(u)], val[int(u)], test[int(u)] = items[:-2], int(items[-2]), int(items[-1])
    return train, val, test


path = CONFIG["ml1m_ratings_path"]
if path:
    raw, source = load_ml1m(path), "MovieLens-1M"
else:
    raw, source = make_synthetic(), "synthetic (group-structured)"

raw = five_core(raw, k=5)
raw, N_USERS, N_ITEMS = reindex(raw)
TRAIN, VAL, TEST = temporal_split(raw)

# Tie-break determinism: re-running the split must give an identical test assignment.
_, _, test_again = temporal_split(raw)
tie_ok = TEST == test_again
record_gate("temporal_split_deterministic", tie_ok,
            f"{'identical' if tie_ok else 'DIFFERENT'} test items across two runs of the split")

print(f"\nsource            : {source}")
print(f"users / items     : {N_USERS} / {N_ITEMS}")
print(f"interactions      : {len(raw)}")
print(f"density           : {len(raw) / (N_USERS * N_ITEMS):.4%}")
print(f"users with splits : {len(TRAIN)}")
print(f"median history    : {np.median([len(h) for h in TRAIN.values()]):.0f}")

## Part 2 — Two model families, and why the original spec picked the wrong one

This is correction 1, and it is the reason the notebook exists.

The player set is the user's retained interactions, and the intervention masks or downweights them **without retraining**. For the characteristic function `v_u(S)` to vary with `S` at all, the model's score must read the retained history *at inference time*.

`StaticMF` below is textbook BPR-MF. Its user representation `P[u]` is a free parameter fixed during training, and `score` takes a `history` argument purely to satisfy the shared interface — it never reads it. This is not a strawman implementation; it is what BPR-MF is. LightGCN behaves the same way, since its user embedding is produced by message passing at training time and recomputing it from a subset of edges would be retraining.

`ProfileAgg` computes the user vector at scoring time as a weighted mean of the retained history's item embeddings, so masking is `w -> 0` and the bounded downweighting of spec §8 is `w -> rho*w`. Both have an exact algebraic meaning with no retraining.

Part 3 measures the difference. The expected result is that `StaticMF` shows **exactly zero** sensitivity — not small, zero — and that is what makes it a useful control: it proves the test has power rather than merely reporting a small number.

In [ ]:
class StaticMF:
    """BPR-MF. P[u] is a trained free parameter; `history` is accepted and ignored.

    Included as the negative control for the masking gate. Under the spec's player
    definition this model makes v_u(S) constant in S, so every Shapley value is zero.
    """

    history_conditioned = False
    label = "StaticMF (BPR-MF)"

    def __init__(self, n_users, n_items, d, seed=SEED):
        r = np.random.default_rng(seed)
        self.P = 0.1 * r.standard_normal((n_users, d))
        self.Q = 0.1 * r.standard_normal((n_items, d))
        self.n_items = n_items

    def fit(self, train, epochs=40, lr=0.05, reg=0.01, seed=SEED):
        r = np.random.default_rng(seed)
        users = np.array(sorted(train))
        pos = {u: set(train[u].tolist()) for u in users}
        for _ in range(epochs):
            r.shuffle(users)
            for u in users:
                h = train[u]
                i = int(h[r.integers(h.size)])
                j = int(r.integers(self.n_items))
                while j in pos[u]:
                    j = int(r.integers(self.n_items))
                pu, qi, qj = self.P[u].copy(), self.Q[i].copy(), self.Q[j].copy()
                g = sigmoid(-(pu @ (qi - qj)))
                self.P[u] += lr * (g * (qi - qj) - reg * pu)
                self.Q[i] += lr * (g * pu - reg * qi)
                self.Q[j] += lr * (-g * pu - reg * qj)
        return self

    def score(self, u, items, history=None, weights=None):
        return self.Q[items] @ self.P[u]


class ProfileAgg:
    """History-conditioned. The user vector is a weighted mean of retained item vectors."""

    history_conditioned = True
    label = "ProfileAgg (history-conditioned)"

    def __init__(self, n_items, d, seed=SEED):
        r = np.random.default_rng(seed)
        self.Q = 0.1 * r.standard_normal((n_items, d))
        self.d, self.n_items = d, n_items

    def user_vector(self, history, weights=None):
        history = np.asarray(history, dtype=int)
        if history.size == 0:
            return np.zeros(self.d)          # spec 7.4: empty coalition -> zero profile
        w = np.ones(history.size) if weights is None else np.asarray(weights, dtype=float)
        total = w.sum()
        if total <= 1e-12:
            return np.zeros(self.d)
        return (self.Q[history] * w[:, None]).sum(0) / total

    def score(self, u, items, history, weights=None):
        return self.Q[items] @ self.user_vector(history, weights)

    def fit(self, train, epochs=40, lr=0.05, reg=0.01, seed=SEED):
        r = np.random.default_rng(seed)
        users = np.array(sorted(train))
        pos = {u: set(train[u].tolist()) for u in users}
        for _ in range(epochs):
            r.shuffle(users)
            for u in users:
                h = train[u]
                if h.size < 2:
                    continue
                k = int(r.integers(h.size))
                i, ctx = int(h[k]), np.delete(h, k)   # leave the positive out of its own profile
                if ctx.size == 0:
                    continue
                j = int(r.integers(self.n_items))
                while j in pos[u]:
                    j = int(r.integers(self.n_items))
                p = self.Q[ctx].mean(0)
                qi, qj = self.Q[i].copy(), self.Q[j].copy()
                g = sigmoid(-(p @ (qi - qj)))
                self.Q[i] += lr * (g * p - reg * qi)
                self.Q[j] += lr * (-g * p - reg * qj)
                self.Q[ctx] += lr * (g * (qi - qj) / ctx.size - reg * self.Q[ctx])
        return self


t0 = time.time()
static_model = StaticMF(N_USERS, N_ITEMS, CONFIG["n_factors"]).fit(
    TRAIN, epochs=CONFIG["train_epochs"])
profile_model = ProfileAgg(N_ITEMS, CONFIG["n_factors"]).fit(
    TRAIN, epochs=CONFIG["train_epochs"])
print(f"trained both models in {time.time() - t0:.1f}s")

## Part 3 — The gate (spec §7.1.1)

Candidate sets are built first, because everything downstream must be scored against a set that is fixed once and reused for every coalition, every intervention, and every attribution method. Otherwise a change in retrieval gets mistaken for an attribution effect.

**A simplification to be aware of:** this notebook samples negatives and inserts the held-out item, so candidate recall is 1.0 by construction. The real pipeline must retrieve candidates from the frozen model and *report* recall, because in that setting users whose test item is not retrieved score zero for every coalition and cannot contribute to a ranking-improvement claim. That distinction does not affect any gate here, all of which are structural.

Ranking uses `lexsort` with a fixed per-user random tie-break vector. This matters at the empty coalition, where every score is zero and the ordering would otherwise be whatever `argsort` happens to do.

The gate itself: mask one uniformly chosen history item per user, rescore, and measure whether the top-10 moved.

In [ ]:
@dataclass
class UserGame:
    """Everything needed to evaluate v_u(S) for one user, all frozen up front."""
    u: int
    players: np.ndarray     # the retained history: one player per interaction
    cands: np.ndarray       # fixed candidate set
    tie: np.ndarray         # fixed tie-break key, one per candidate
    target_pos: int         # index of the held-out item inside `cands`


def build_candidates(train, test, n_items, size, seed=SEED):
    r = np.random.default_rng(seed)
    longest = max(len(h) for h in train.values())
    if n_items <= size + longest:
        raise ValueError(
            f"catalogue too small: {n_items} items cannot supply {size} candidates "
            f"disjoint from a history of up to {longest}. Lower CONFIG['candidate_size']."
        )
    cands, ties = {}, {}
    for u in sorted(train):
        seen = set(train[u].tolist()) | {test[u]}
        pool = []
        while len(pool) < size - 1:
            for b in r.integers(0, n_items, size=size):
                b = int(b)
                if b not in seen:
                    pool.append(b)
                    seen.add(b)
                    if len(pool) == size - 1:
                        break
        c = np.array([test[u]] + pool, dtype=int)
        c = c[r.permutation(c.size)]
        cands[u], ties[u] = c, r.random(c.size)
    return cands, ties


def ndcg_single(scores, tie, target_pos, k):
    """NDCG@k with exactly one relevant item. Ties broken by the frozen key."""
    order = np.lexsort((tie, -scores))
    rank = int(np.flatnonzero(order == target_pos)[0]) + 1
    return 1.0 / math.log2(rank + 1) if rank <= k else 0.0


def build_games(train, test, cands, ties, n_max):
    games = {}
    for u in sorted(train):
        hist = train[u][-n_max:]            # most recent n_max interactions (spec 6)
        if hist.size < 4:
            continue
        games[u] = UserGame(u, hist, cands[u], ties[u],
                            int(np.flatnonzero(cands[u] == test[u])[0]))
    return games


def utility(model, g: UserGame, mask=None, weights=None, k=None):
    """v_u(S). `mask` selects the coalition; `weights` applies rho-downweighting."""
    k = k or CONFIG["ndcg_k"]
    hist = g.players if mask is None else g.players[mask]
    w = weights if mask is None else (None if weights is None else weights[mask])
    return ndcg_single(model.score(g.u, g.cands, hist, w), g.tie, g.target_pos, k)


CANDS, TIES = build_candidates(TRAIN, TEST, N_ITEMS, CONFIG["candidate_size"])
GAMES = build_games(TRAIN, TEST, CANDS, TIES, CONFIG["n_max_history"])
print(f"games built for {len(GAMES)} users\n")


def masking_sensitivity(model, games, n_users, seed=SEED):
    """Mask one random history item per user; report how often the top-10 moves."""
    r = np.random.default_rng(seed)
    users = sorted(games)[:n_users]
    changed, deltas = 0, []
    for u in users:
        g = games[u]
        n = g.players.size
        full = np.ones(n, dtype=bool)
        base_scores = model.score(g.u, g.cands, g.players, None)
        base_top = g.cands[np.lexsort((g.tie, -base_scores))][:CONFIG["ndcg_k"]]
        base_ndcg = ndcg_single(base_scores, g.tie, g.target_pos, CONFIG["ndcg_k"])

        mask = full.copy()
        mask[int(r.integers(n))] = False
        new_scores = model.score(g.u, g.cands, g.players[mask], None)
        new_top = g.cands[np.lexsort((g.tie, -new_scores))][:CONFIG["ndcg_k"]]
        new_ndcg = ndcg_single(new_scores, g.tie, g.target_pos, CONFIG["ndcg_k"])

        changed += int(not np.array_equal(base_top, new_top))
        deltas.append(abs(new_ndcg - base_ndcg))
    return changed / len(users), float(np.mean(deltas)), len(users)


rows = []
for m in (static_model, profile_model):
    frac, mad, n = masking_sensitivity(m, GAMES, CONFIG["n_gate_users"])
    rows.append({"model": m.label, "history_conditioned": m.history_conditioned,
                 "users": n, "frac_top10_changed": frac, "mean_abs_dNDCG": mad})
sens = pd.DataFrame(rows)
display(sens.style.format({"frac_top10_changed": "{:.3f}", "mean_abs_dNDCG": "{:.6f}"}))

prof = sens[~sens["history_conditioned"].eq(False)].iloc[0]
stat = sens[sens["history_conditioned"].eq(False)].iloc[0]

record_gate("masking_sensitivity",
            prof["frac_top10_changed"] >= 0.50 and prof["mean_abs_dNDCG"] >= 1e-3,
            f"ProfileAgg moved the top-10 for {prof['frac_top10_changed']:.1%} of users, "
            f"mean |dNDCG| = {prof['mean_abs_dNDCG']:.5f}")

record_gate("static_control_is_inert",
            stat["frac_top10_changed"] == 0.0 and stat["mean_abs_dNDCG"] == 0.0,
            f"StaticMF moved the top-10 for {stat['frac_top10_changed']:.1%} of users "
            f"(expected exactly 0% -- confirms the test has power)")

## Part 4 — Coalition sanity checks (spec §14, tests 1, 2, 3, 11)

Four properties that must hold before any Shapley value is worth computing:

1. **The empty coalition is defined and deterministic.** With a zero profile vector every candidate scores zero, so the ranking is decided entirely by the frozen tie-break. Two calls must agree.
2. **The full coalition reproduces the unmodified recommendation.** If it does not, the masking layer is altering something it should not.
3. **Masking one user's interaction leaves every other user untouched.** Catches shared mutable state, which is easy to introduce when caching profiles.
4. **A zero-budget intervention has zero effect.** The degenerate case of the intervention simulator.

Also verified: the static control produces a *constant* characteristic function, which is the concrete form the design flaw would have taken.

In [ ]:
probe_users = sorted(GAMES)[:50]

# --- test 1: empty coalition defined and deterministic -----------------------
empty_vals = []
for u in probe_users:
    g = GAMES[u]
    m = np.zeros(g.players.size, dtype=bool)
    a, b = utility(profile_model, g, mask=m), utility(profile_model, g, mask=m)
    empty_vals.append((a, b))
empty_det = all(a == b for a, b in empty_vals)
empty_finite = all(np.isfinite(a) for a, _ in empty_vals)
record_gate("empty_coalition_defined", empty_det and empty_finite,
            f"v_u(empty) finite and reproducible; mean = {np.mean([a for a, _ in empty_vals]):.4f}")

# --- test 2: full coalition reproduces the unmodified recommendation ---------
full_ok = True
for u in probe_users:
    g = GAMES[u]
    direct = ndcg_single(profile_model.score(g.u, g.cands, g.players, None),
                         g.tie, g.target_pos, CONFIG["ndcg_k"])
    via_mask = utility(profile_model, g, mask=np.ones(g.players.size, dtype=bool))
    full_ok &= abs(direct - via_mask) < 1e-12
record_gate("full_coalition_matches_base", full_ok,
            "masking layer with all players active reproduces the base recommendation exactly")

# --- test 3: masking one user does not touch another -------------------------
u0, u1 = probe_users[0], probe_users[1]
g0, g1 = GAMES[u0], GAMES[u1]
before = utility(profile_model, g1)
m0 = np.ones(g0.players.size, dtype=bool)
m0[0] = False
_ = utility(profile_model, g0, mask=m0)
after = utility(profile_model, g1)
record_gate("masking_is_user_local", before == after,
            "masking a player for one user left another user's utility unchanged")

# --- test 11: zero-budget intervention has zero effect -----------------------
zero_budget_ok = True
for u in probe_users[:20]:
    g = GAMES[u]
    w = np.ones(g.players.size)          # rho applied to nothing
    zero_budget_ok &= abs(utility(profile_model, g, weights=w) - utility(profile_model, g)) < 1e-12
record_gate("zero_budget_no_effect", zero_budget_ok,
            "an intervention touching no player produced exactly zero change")

# --- the failure mode, made concrete ----------------------------------------
static_spread, profile_spread = [], []
rng_probe = np.random.default_rng(SEED)
for u in probe_users[:20]:
    g = GAMES[u]
    vals_s, vals_p = [], []
    for _ in range(8):
        m = rng_probe.random(g.players.size) < 0.5
        if not m.any():
            m[0] = True
        vals_s.append(utility(static_model, g, mask=m))
        vals_p.append(utility(profile_model, g, mask=m))
    static_spread.append(np.ptp(vals_s))
    profile_spread.append(np.ptp(vals_p))

print(f"\nrange of v_u(S) over 8 random coalitions, averaged over 20 users")
print(f"  StaticMF   : {np.mean(static_spread):.8f}   <- constant characteristic function")
print(f"  ProfileAgg : {np.mean(profile_spread):.8f}")
record_gate("value_function_is_non_constant", np.mean(profile_spread) > 1e-6,
            f"ProfileAgg v_u(S) varies across coalitions (mean range "
            f"{np.mean(profile_spread):.5f}); StaticMF is flat at {np.mean(static_spread):.1e}")

## Part 5 — Monte Carlo Shapley, and why the efficiency diagnostic is vacuous

This is correction 3.

Two estimators are implemented because they behave differently on the one diagnostic the spec relies on.

**Prefix walk.** Draw a permutation, add players one at a time, take consecutive differences of `v_u`. Costs `n+1` evaluations per permutation. The marginal contributions telescope, so

```
sum_p phi_p  =  v(full) - v(empty)
```

holds **exactly for every individual permutation**, and therefore exactly in the mean — regardless of how few permutations were drawn or how far from converged the estimate is.

**Independent marginals.** Sample each player's predecessor set from its own permutation. Costs roughly `2n` evaluations and the telescoping is broken, so efficiency error is genuinely informative.

The original spec asked for the prefix walk (implicitly, via paired marginal evaluation and caching) *and* listed efficiency error in the acceptance criteria as evidence of estimate quality. Those two things are incompatible: under the prefix walk the reported error is floating-point noise at `1e-16` whatever you do, so it certifies nothing. The cell below measures both estimators at a deliberately inadequate `M=3` to show that the prefix walk's efficiency error stays at machine precision while its estimate is still visibly unconverged.

Also here: the synthetic additive game, where the correct Shapley values are known in closed form, plus the symmetry check.

In [ ]:
def shapley_prefix(n, v_fn, M, seed, antithetic=True):
    """Permutation prefix walk. n+1 evaluations per permutation; efficiency telescopes."""
    r = np.random.default_rng(seed)
    cache = {}

    def v(key):
        if key not in cache:
            cache[key] = v_fn(key)
        return cache[key]

    perms = []
    for _ in range(M):
        p = r.permutation(n)
        perms.append(p)
        if antithetic:
            perms.append(p[::-1].copy())

    phi = np.zeros(n)
    for p in perms:
        cur, prev = [], v(())
        for j in p:
            cur.append(int(j))
            val = v(tuple(sorted(cur)))
            phi[j] += val - prev
            prev = val
    return phi / len(perms), len(cache)


def shapley_independent(n, v_fn, M, seed):
    """Each player's predecessor set drawn from its own permutation. Efficiency not exact."""
    r = np.random.default_rng(seed)
    cache = {}

    def v(key):
        if key not in cache:
            cache[key] = v_fn(key)
        return cache[key]

    phi = np.zeros(n)
    for j in range(n):
        acc = 0.0
        for _ in range(M):
            p = r.permutation(n)
            pre = sorted(int(x) for x in p[:int(np.flatnonzero(p == j)[0])])
            acc += v(tuple(sorted(pre + [j]))) - v(tuple(pre))
        phi[j] = acc / M
    return phi, len(cache)


def game_v_fn(model, g):
    n = g.players.size

    def v_fn(key):
        mask = np.zeros(n, dtype=bool)
        if key:
            mask[list(key)] = True
        return utility(model, g, mask=mask)

    return v_fn


# --- synthetic additive game: closed-form ground truth ----------------------
rng_syn = np.random.default_rng(SEED)
a_true = rng_syn.normal(size=8)
a_true[6] = a_true[5]                       # identical players -> symmetry check
phi_syn, _ = shapley_prefix(8, lambda key: float(a_true[list(key)].sum()) if key else 0.0,
                            M=40, seed=SEED)
record_gate("synthetic_additive_game", np.allclose(phi_syn, a_true, atol=1e-9),
            f"max deviation from closed form = {np.max(np.abs(phi_syn - a_true)):.2e}")
record_gate("symmetry_identical_players", abs(phi_syn[5] - phi_syn[6]) < 1e-9,
            f"two identical players received phi differing by {abs(phi_syn[5] - phi_syn[6]):.2e}")

# --- efficiency under both estimators, at a deliberately tiny M -------------
eval_users = sorted(GAMES)[:CONFIG["n_eval_users"]]
eff_rows = []
for u in eval_users[:15]:
    g = GAMES[u]
    n = g.players.size
    v_fn = game_v_fn(profile_model, g)
    total = v_fn(tuple(range(n))) - v_fn(())
    p_pre, _ = shapley_prefix(n, v_fn, M=3, seed=SEED + u)
    p_ind, _ = shapley_independent(n, v_fn, M=3, seed=SEED + u)
    eff_rows.append({"prefix_walk": abs(p_pre.sum() - total),
                     "independent_marginals": abs(p_ind.sum() - total)})
eff = pd.DataFrame(eff_rows).mean()
print("\nmean |sum(phi) - (v(full) - v(empty))| at M=3 (deliberately unconverged):")
print(f"  prefix walk           : {eff['prefix_walk']:.3e}   <- an identity, not a result")
print(f"  independent marginals : {eff['independent_marginals']:.3e}")
record_gate("efficiency_diagnostic_is_vacuous_under_prefix_walk",
            eff["prefix_walk"] < 1e-12 < eff["independent_marginals"],
            "prefix-walk efficiency error is machine precision even at M=3, so it cannot "
            "certify convergence; use the spec 9 convergence criterion instead")

# --- the real Shapley values, at the configured M ---------------------------
t0 = time.time()
PHI = {}
for u in eval_users:
    g = GAMES[u]
    PHI[u], _ = shapley_prefix(g.players.size, game_v_fn(profile_model, g),
                               M=CONFIG["mc_permutations"], seed=SEED + u)
print(f"\nMC Shapley for {len(PHI)} users in {time.time() - t0:.1f}s "
      f"(M={CONFIG['mc_permutations']}, antithetic)")

## Part 6 — Interventions and the oracle `a*` (spec §11.4)

This is correction 5. The original spec defined regret against "the best feasible action under the declared budget" without saying how that best action is found, which is fine at `B=1` and combinatorially impossible at `B=3`.

With `n_max=20` players and three intervention strengths:

- `B=1` — 20 x 3 = 60 evaluations per user. Exhaustive, trivial.
- `B=3` — C(20,3) x 3^3 = 30,780 per user. Exhaustive is out.

So `a*` is exhaustive at `B=1` and **greedy forward selection** above that, and the paper must say so, because greedy regret is a lower bound on true regret rather than the real thing. The cell below quantifies the gap by running both on a restricted setting (`B=2`, players capped at 10) where exhaustive search is still affordable — that measured gap is what licenses the greedy approximation in the main experiment.

In [ ]:
def base_utility(model, g):
    return utility(model, g)


def apply_action(model, g, action, base):
    """action = {player_index: rho}. Returns the signed effect on NDCG."""
    w = np.ones(g.players.size)
    for j, rho in action.items():
        w[j] = rho
    return utility(model, g, weights=w) - base


def single_player_effects(model, g, rho=0.0):
    """Delta_u(p) for every player at one intervention strength."""
    base = base_utility(model, g)
    out = np.empty(g.players.size)
    for j in range(g.players.size):
        out[j] = apply_action(model, g, {j: rho}, base)
    return out, base


def astar_exhaustive(model, g, budget, rho_grid, players=None):
    """Exact oracle. Only tractable for small budget/player counts."""
    base = base_utility(model, g)
    players = range(g.players.size) if players is None else players
    best, best_act = -np.inf, None
    for combo in combinations(players, budget):
        for rhos in np.ndindex(*([len(rho_grid)] * budget)):
            act = {p: rho_grid[r] for p, r in zip(combo, rhos)}
            d = apply_action(model, g, act, base)
            if d > best:
                best, best_act = d, act
    return best, best_act


def astar_greedy(model, g, budget, rho_grid, players=None):
    """Greedy forward selection: pick, fix, re-measure against the modified profile."""
    base = base_utility(model, g)
    players = list(range(g.players.size)) if players is None else list(players)
    chosen, remaining = {}, set(players)
    best = 0.0
    for _ in range(budget):
        step_best, step_act = -np.inf, None
        for p in remaining:
            for rho in rho_grid:
                cand = dict(chosen)
                cand[p] = rho
                d = apply_action(model, g, cand, base)
                if d > step_best:
                    step_best, step_act = d, (p, rho)
        if step_act is None:
            break
        chosen[step_act[0]] = step_act[1]
        remaining.discard(step_act[0])
        best = step_best
    return best, chosen


rho_grid = CONFIG["rho_grid"]

# B=1: exhaustive is affordable, so the oracle is exact.
t0 = time.time()
ASTAR_B1 = {u: astar_exhaustive(profile_model, GAMES[u], 1, rho_grid) for u in eval_users}
record_gate("astar_exhaustive_B1", True,
            f"exact oracle for B=1 over {len(eval_users)} users in {time.time() - t0:.1f}s")

# B=2 on a restricted player set: quantify how much greedy loses.
gaps = []
for u in eval_users[:20]:
    g = GAMES[u]
    sub = list(range(min(10, g.players.size)))
    ex, _ = astar_exhaustive(profile_model, g, 2, rho_grid, players=sub)
    gr, _ = astar_greedy(profile_model, g, 2, rho_grid, players=sub)
    gaps.append(ex - gr)
gaps = np.array(gaps)
print(f"\ngreedy vs exhaustive a* at B=2 (players capped at 10, {len(gaps)} users)")
print(f"  mean shortfall : {gaps.mean():.5f} NDCG")
print(f"  max shortfall  : {gaps.max():.5f} NDCG")
print(f"  exact matches  : {(gaps <= 1e-12).mean():.1%} of users")
record_gate("greedy_astar_gap_quantified", bool(np.all(gaps >= -1e-12)),
            f"greedy never beat exhaustive (as expected); mean shortfall {gaps.mean():.5f} NDCG, "
            f"exact on {(gaps <= 1e-12).mean():.0%} of users")

## Part 7 — AIA, its null, and the `B=1` circularity

Two corrections land here.

**Correction 4 — AIA needs a null.** An AIA value means nothing on its own. The earlier cross-domain ActionShap runs produced an AIA of 0.518 for a *random* attribution, which was alarming only because there was no reference distribution to compare it against. The null is built by shuffling the measured effects across players **within each user** and recomputing, which respects the fact that Spearman's null distribution depends on the player count `n_u`, and `n_u` varies across users.

**Correction 7 — RQ2 is circular at `B=1`.** Work the algebra:

```
Delta_u(p) = v_u(full without p) - v_u(full)        <- the measured intervention effect
LOO_u(p)   = v_u(full) - v_u(full without p)        <- the ablation attribution
```

These are the same number negated, so `|LOO| == |Delta|` identically and `AIA(LOO) = 1.0` for every user, by construction rather than by merit. The cell below confirms this numerically — it should print exactly 1.000.

That result is not a finding, it is an identity, and it means leave-one-out must be reported as the **oracle** at `B=1` rather than as a competing method. The real comparison has to sit at `B>=2`, where ablation's implicit additivity assumption breaks on interacting players and a coalition-aware estimator has something to win. That comparison is the last block below.

In [ ]:
def aia(phi, delta):
    """Spearman between |attribution| and |measured effect|. NaN if either is constant."""
    a, b = np.abs(np.asarray(phi)), np.abs(np.asarray(delta))
    if a.std() < 1e-12 or b.std() < 1e-12:
        return np.nan
    return float(spearmanr(a, b)[0])


def aia_null(phi, delta, draws, rng):
    """Within-user permutation null: shuffle the effects across players."""
    d, out = np.abs(np.asarray(delta)).copy(), []
    for _ in range(draws):
        rng.shuffle(d)
        v = aia(phi, d)
        if not np.isnan(v):
            out.append(v)
    return np.asarray(out)


# Ground-truth single-player effects, and the four attribution methods.
rng_m = np.random.default_rng(SEED)
DELTA, ATTR = {}, {"shapley": {}, "loo": {}, "recency": {}, "random": {}}
for u in eval_users:
    g = GAMES[u]
    d, base = single_player_effects(profile_model, g, rho=0.0)
    DELTA[u] = d
    ATTR["shapley"][u] = PHI[u]
    ATTR["loo"][u] = -d                              # v(full) - v(full \ p)
    ATTR["recency"][u] = np.arange(g.players.size, dtype=float)
    ATTR["random"][u] = rng_m.normal(size=g.players.size)

rng_null = np.random.default_rng(SEED)
rows = []
for name, attr in ATTR.items():
    obs = np.array([aia(attr[u], DELTA[u]) for u in eval_users], dtype=float)
    obs = obs[~np.isnan(obs)]
    null = np.concatenate([
        aia_null(attr[u], DELTA[u], max(1, CONFIG["null_draws"] // len(eval_users)), rng_null)
        for u in eval_users
    ])
    rows.append({
        "method": name,
        "AIA": obs.mean(),
        "null_mean": null.mean(),
        "null_p95": np.percentile(null, 95),
        "p_value": float((null.mean() >= obs.mean()) if obs.size == 0
                         else (null >= obs.mean()).mean()),
    })
aia_tbl = pd.DataFrame(rows).set_index("method")
print("AIA against the within-user permutation null (B=1, single-player masking)\n")
display(aia_tbl.style.format("{:.4f}"))

record_gate("aia_null_centres_at_zero", abs(aia_tbl["null_mean"].mean()) < 0.05,
            f"permutation null mean = {aia_tbl['null_mean'].mean():+.4f} (must be near 0, "
            f"otherwise the metric is mis-specified)")

record_gate("B1_is_circular_for_LOO", abs(aia_tbl.loc["loo", "AIA"] - 1.0) < 1e-9,
            f"AIA(LOO) = {aia_tbl.loc['loo', 'AIA']:.6f} at B=1 -- an identity, so LOO is the "
            f"oracle here and RQ2 must be evaluated at B>=2")

# --- the comparison that is NOT circular: joint interventions at B=2 --------
def select_top_b(scores, b):
    return list(np.argsort(-np.abs(np.asarray(scores)))[:b])


B = 2
rows = []
for name, attr in ATTR.items():
    achieved, oracle = [], []
    for u in eval_users[:25]:
        g = GAMES[u]
        base = base_utility(profile_model, g)
        sub = list(range(min(10, g.players.size)))
        picks = [p for p in select_top_b(attr[u][sub], B)]
        achieved.append(apply_action(profile_model, g, {p: 0.0 for p in picks}, base))
        oracle.append(astar_exhaustive(profile_model, g, B, (0.0,), players=sub)[0])
    achieved, oracle = np.array(achieved), np.array(oracle)
    rows.append({"method": name, "achieved_dNDCG": achieved.mean(),
                 "oracle_dNDCG": oracle.mean(), "regret": (oracle - achieved).mean()})
joint = pd.DataFrame(rows).set_index("method").sort_values("regret")
print(f"\nJoint intervention at B={B} (masking two players; oracle is exhaustive)\n")
display(joint.style.format("{:.5f}"))

sep = joint.loc["loo", "regret"] - joint.loc["shapley", "regret"]
record_gate("B2_comparison_is_informative",
            joint["regret"].std() > 1e-9,
            f"methods separate at B={B} (regret spread {joint['regret'].std():.5f}); "
            f"shapley minus loo regret = {sep:+.5f}")

## Part 8 — Verdict

`masking_sensitivity` and `static_control_is_inert` are the two that decide whether the project proceeds. The rest confirm that the corrected metric and estimator definitions behave as the revised spec claims.

Read `B1_is_circular_for_LOO` as a **confirmation**, not a failure: it passing means the algebra behaves as predicted and the RQ2 redesign was necessary.

In [ ]:
summary = pd.DataFrame(
    [{"gate": k, "result": "PASS" if v[0] else "FAIL", "detail": v[1]}
     for k, v in GATES.items()]
).set_index("gate")

BLOCKING = ["masking_sensitivity", "static_control_is_inert", "value_function_is_non_constant"]

print("=" * 100)
print("GATE SUMMARY".center(100))
print("=" * 100)
with pd.option_context("display.max_colwidth", 90, "display.width", 200):
    print(summary.to_string())
print("=" * 100)

failed = [k for k, (ok, _) in GATES.items() if not ok]
blocking_failed = [k for k in failed if k in BLOCKING]

if blocking_failed:
    print(f"\nSTOP. Blocking gate(s) failed: {', '.join(blocking_failed)}")
    print("The model is not history-conditioned, so the player definition in spec section 6")
    print("cannot be paired with the intervention semantics in section 8. Change the model")
    print("family before writing any further pipeline code.")
elif failed:
    print(f"\nPROCEED WITH CARE. Non-blocking gate(s) failed: {', '.join(failed)}")
else:
    print("\nALL GATES PASSED. The revised design is buildable as specified.")
    print("Next: spec section 17 build order, step 1 (data loader and frozen temporal split).")

try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    axes[0].bar(sens["model"], sens["frac_top10_changed"], color=["#bbb", "#2b6cb0"])
    axes[0].axhline(0.5, ls="--", c="crimson", lw=1, label="gate threshold")
    axes[0].set_ylabel("fraction of users whose top-10 moved")
    axes[0].set_title("Masking sensitivity (spec 7.1.1)")
    axes[0].tick_params(axis="x", labelsize=8)
    axes[0].legend(fontsize=8)

    order = aia_tbl.index.tolist()
    axes[1].bar(order, aia_tbl["AIA"], color="#2b6cb0", label="observed AIA")
    axes[1].plot(order, aia_tbl["null_p95"], "o--", c="crimson", lw=1,
                 ms=4, label="null 95th pct")
    axes[1].axhline(0, c="k", lw=0.8)
    axes[1].set_ylabel("AIA")
    axes[1].set_title("AIA vs permutation null (B=1)")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()
except ImportError:
    print("\n(matplotlib not installed -- skipping plots)")